In [ ]:
# Imports
import cProfile
import pstats
import matplotlib.pyplot as plt
from regions import Regions
from astropy import units as u
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization.wcsaxes import WCSAxes
from astropy.coordinates import SkyCoord, FK5
from spectral_cube import SpectralCube
from velocity_tools import extract_streamline, gradient_descent, stream_lines_grad
from velocity_tools import stream_lines # won't use this directly, but needed to compare with stream_lines_grad
import os
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax import value_and_grad
import pandas as pd
import optax

import warnings
warnings.filterwarnings('ignore', message='.*PV2_1.*')
warnings.filterwarnings('ignore', message='.*PV2_2.*')
warnings.filterwarnings('ignore', message='.*TIMESYS.*')

# Settings
hltau_c= SkyCoord("4h31m38.43s", "+18d13m57.19s", frame='fk5')
hltau_ref = hltau_c.skyoffset_frame()
iras2a_c = SkyCoord("3h28m55.569s", "+31d14m37.025s", frame='fk5')
iras2a_ref = iras2a_c.skyoffset_frame()
b5irs1_c = SkyCoord("3h47m41.577s", "+32d51m43.745s", frame='fk5')
b5irs1_ref = b5irs1_c.skyoffset_frame()
distance_hltau = 147 #parsecs
distance_iras2a = 293 #parsecs
distance_b5irs1 = 302 #parsecs
# choose which distance
distance = distance_b5irs1

# cubefile = 'test_data/HLTau/HLTAU_HCOp32.fits'
# file_Tpeak = 'test_data/HLTau/HLTAU_HCOp32_Tpeak.fits'
#cubefile = '../test_data/B5IRS1/B5-IRS1_CD_c-HCCCH_6_0_6_5_1_5_sc_contsub_sm-merged-pbcor.fits'
cubefile = '../test_data/B5IRS1/components_blueshifted_envelope_ppv.fits'

# some constants
G = 6.67430e-11 * (1e-3)**2 * (1.988416e30) / (1.4959787e11) # in au (km/s)^2 * Msol^-1
au_in_km = 1.4959787e8 #km



### Original cube with spectra: Prepare the 1D streamer emission from the cube

In [ ]:
# get the spectralcube object from the data using spectral-cube
hdu = fits.open(cubefile)[0]
cube = SpectralCube.read(hdu).with_spectral_unit(u.km/u.s, rest_value=hdu.header['RESTFREQ']*u.Hz)

# # mask the cube using the region file
# region_file = 'test_data/B5IRS1/streamer_region_small.reg'
# regions = Regions.read(region_file, format='ds9')
# # # create spatial mask from first region
# reg = regions[0]
# # # make 2D mask in celestial plane
# mask2d = reg.to_pixel(cube.wcs.celestial).to_mask()
# # # convert to image-sized boolean array
# spatial_mask = mask2d.to_image(cube.shape[1:]).astype(bool)
# # # expand to 3D for spectral cube
# mask3d = np.broadcast_to(spatial_mask, cube.shape)
# # # apply mask
# cube = cube.with_mask(mask3d)


# TODO: this streamer extraction should be replaced with clustering-based streamer extraction

# extract the subcube with the streamer (we got this from the tipsy tutorial, no need to plot)
## Limits for extracting subcube with streamer

'''
vmin = 7    # Min. vel. of streamer
vmax = 10
xmin = -3   # Min R.A. offset (arcsec) to consider for streamer
xmax = -1
ymin = -3   # Min. Decl. offset (arcsec) to consider for streamer
ymax = 0.5
rms_thresh = 4  # sigma threshold for streamer
'''

vmin = 8
vmax = 11
# if it's a clustering-based cube already, you don't need this
#rms_thresh = 2

# extract the streamer subcube
info_header = cube.header  # header of the cube, contains required information
vunit = cube.spectral_axis.unit
## Note: a check can be added to see if requested limits are within the limits of the cube itself


streamer_cubev = cube.spectral_slab(vmin*vunit,vmax*vunit)    # Selecting velocities

# if it's a clustering-based cube already, you don't need to mask for rms
#streamer_cube = streamer_cubev.with_mask(streamer_cubev > rms_thresh*streamer_cubev.mad_std())  # Removing low flux values 
streamer_cube = streamer_cubev


print('Streamer cube shape:', streamer_cube.shape)


In [ ]:
from velocity_tools.streamfit.extract_streamline import cartesian_to_polar, get_distance_metric


n_points = 10 # the number of points we want to reduce the data to


# try a method binning by polar angle instead
def get_polar_angle_metric(ra_coords, dec_coords):

    pc_r, pc_theta = cartesian_to_polar(ra_coords, dec_coords)
    polar_angle_metric = pc_theta
    return polar_angle_metric

def reduce_to_1D_polar(streamer_cube, n_elements=10):
    '''
    This function will reduce a cube of emission to a 1D 'streamline', 
    by weighted means in polar angle bins.

    In this function, 'pc' is short for point cloud.

    Parameters
    ----------
    streamer_cube : SpectralCube object, should contain only streamer emission
    n_elements : int, number of elements to reduce the cube to


    Returns
    -------
    pc_means : array of shape (3, n_elements), the weighted mean coordinates of each bin
    index 0 = RA offsets (arcsec)
    index 1 = Dec offsets (arcsec)
    index 2 = velocity (km/s)
    '''
    print('Starting reduction')
    nz, ny, nx = streamer_cube.shape

    # create coordinate arrays for RA and Dec in arcsec without assuming the input celestial units
    y_indices, x_indices = np.mgrid[0:ny, 0:nx]
    world_coords = streamer_cube.wcs.celestial.pixel_to_world_values(x_indices.ravel(), y_indices.ravel())
    ra_unit = u.Unit(streamer_cube.header.get('CUNIT1', streamer_cube.wcs.celestial.world_axis_units[0]))
    dec_unit = u.Unit(streamer_cube.header.get('CUNIT2', streamer_cube.wcs.celestial.world_axis_units[1]))
    ra_ref = streamer_cube.header['CRVAL1'] * ra_unit
    dec_ref = streamer_cube.header['CRVAL2'] * dec_unit
    ra_coords = ((world_coords[0].reshape(ny, nx) * ra_unit) - ra_ref).to(u.arcsec).value
    ra_coords = ra_coords * np.cos(dec_ref.to(u.rad).value) # cos(dec) correct for declination. in arcsec
    dec_coords = ((world_coords[1].reshape(ny, nx) * dec_unit) - dec_ref).to(u.arcsec).value # in arcsec

    # create velocity array relative to the reference channel, then express it in km/s
    spectral_unit = u.Unit(streamer_cube.header.get('CUNIT3', streamer_cube.spectral_axis.unit))
    spectral_axis = streamer_cube.spectral_axis.to(spectral_unit)
    v_ref = streamer_cube.header['CRVAL3'] * spectral_unit
    v_coords = (spectral_axis - v_ref).to(u.km / u.s).value

    print('Created coordinate arrays')

    # get data and mask
    pcloud = np.array(streamer_cube)
    rms_mask = ~np.isnan(pcloud)
    flux = pcloud[rms_mask]

    # get indices of valid points in pc
    pc_indices = np.indices(pcloud.shape) # indices of all points in pc
    pc_z = pc_indices[0][rms_mask] # z indices of points in pc
    pc_y = pc_indices[1][rms_mask] # y indices of points in pc
    pc_x = pc_indices[2][rms_mask] # x indices of points in pc

    print('Got point cloud with', len(flux), 'points')

    # extract coordinates of valid points using the arrays above
    pc_ra = ra_coords[pc_y, pc_x]
    pc_dec = dec_coords[pc_y, pc_x]
    pc_v = v_coords[pc_z]
    pc_coords = np.array([pc_ra, pc_dec, pc_v]) # shape (3, n_points)   

    # compute polar angle metric to bin the point cloud
    polar_angle_metric = get_polar_angle_metric(pc_coords[0], pc_coords[1])
    print('Computed polar angle metric:', polar_angle_metric)
    b_per = np.linspace(0, 100, n_elements+1) # percentiles to bin the pc into
    partitions = np.array([np.percentile(polar_angle_metric, per) for per in b_per])

    print("Partition boundaries for polar angle metric:", np.round(partitions, 3))

    # take flux-weighted means and stds in each bin
    pc_means = np.zeros((3, n_elements))
    pc_stds = np.zeros((3, n_elements))
    for i in range(n_elements):
        # identify points in this bin, add weighted means and weighted stds
        angle_indices = (polar_angle_metric > partitions[i]) & (polar_angle_metric <= partitions[i+1])
        pc_means[:, i] = np.average(pc_coords.T[angle_indices],
                                 axis=0,
                                 weights=flux[angle_indices])
        pc_stds[:, i] = np.sqrt(np.average((pc_coords.T[angle_indices] - pc_means[:, i])**2,
                                         axis=0,
                                         weights=flux[angle_indices]))
    
    # flip arrays so that they go from large to small distance (towards star)
    pc_means = pc_means[:, ::-1]
    pc_stds = pc_stds[:, ::-1]
        
    
    return pc_coords, pc_means, pc_stds, partitions


# Extract 1D streamline from the data cube - original method
pc_coords, pc_means, pc_stds, partitions = reduce_to_1D_polar(streamer_cube, n_elements=n_points)

In [ ]:
# Prepare data for gradient descent
ra_data = pc_means[0] # offsets in arcsec
dec_data = pc_means[1] # offsets in arcsec
v_data = pc_means[2]   # velocities in km/s (rel to vlsr)
print(f"data velocities (km/s): {v_data}")
print(f"data RA offsets (arcsec): {ra_data}")
print(f"data Dec offsets (arcsec): {dec_data}")
rproj = np.sqrt(ra_data**2 + dec_data**2)
print(f"projected distances from star (arcsec): {rproj}")

ra_sigma = pc_stds[0]
dec_sigma = pc_stds[1]
v_sigma = pc_stds[2]


data = (ra_data, dec_data, v_data)
uncertainties = (ra_sigma, dec_sigma, v_sigma)

### !Monkey-patch of matching metric for angular bins!
Local, does not change source code

In [ ]:
# Monkey-patch the matching metric so fit_streamline matches in angular space instead of radial space.
# This stays local to the notebook session and does not modify the package source.
_original_get_distance_metric = extract_streamline.get_distance_metric

_theta_reference = float(np.median(partitions))

def get_angular_distance_metric(ra_coords, dec_coords, return_trace=False):
    _r_proj, theta_proj = extract_streamline.cartesian_to_polar(
        jnp.asarray(ra_coords, dtype=jnp.float64),
        jnp.asarray(dec_coords, dtype=jnp.float64),
    )
    theta_metric = extract_streamline._wrap_to_pi(theta_proj - _theta_reference)

    if return_trace:
        trace = {
            'n_points': int(jnp.asarray(theta_metric).size),
            'theta_reference': float(_theta_reference),
        }
        return theta_metric, trace

    return theta_metric

extract_streamline.get_distance_metric = get_angular_distance_metric
print(f"Patched matching metric to angular bins using theta_ref={_theta_reference:.6f} rad")


In [ ]:
def plot_angle_bin_lines(ax, partitions, color='lightgrey', linewidth=1, alpha=0.5):
    """
    Plot rays showing the polar angle bin edges on an RA-Dec plot.
    Axis limits are preserved after adding the lines.
    
    Parameters
    ----------
    ax : matplotlib axes object
        The axes to plot on
    partitions : array
        Polar angle bin boundaries in radians
    color : str
        Color of the lines
    linewidth : float
        Line width of the lines
    alpha : float
        Transparency of the lines
    """
    import numpy as np
    # Save current axis limits
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    
    # Add rays that start at the star and extend only in the forward angular direction
    x_min, x_max = sorted(xlim)
    y_min, y_max = sorted(ylim)
    for partition in partitions:
        dx = np.cos(partition)
        dy = np.sin(partition)
        ray_lengths = []
        if dx != 0:
            for x_edge in (x_min, x_max):
                t = x_edge / dx
                if t > 0:
                    y_at_edge = t * dy
                    if y_min <= y_at_edge <= y_max:
                        ray_lengths.append(t)
        if dy != 0:
            for y_edge in (y_min, y_max):
                t = y_edge / dy
                if t > 0:
                    x_at_edge = t * dx
                    if x_min <= x_at_edge <= x_max:
                        ray_lengths.append(t)
        if not ray_lengths:
            continue
        t_max = min(ray_lengths)
        x_vals = np.array([0.0, t_max * dx])
        y_vals = np.array([0.0, t_max * dy])
        ax.plot(x_vals, y_vals, color=color, linewidth=linewidth, alpha=alpha)
    
    # Restore original axis limits
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

In [ ]:
# plot it
import matplotlib.pyplot as plt
# plot the observed data points as a scatter
plt.scatter(pc_coords[0], pc_coords[1], s=1, alpha=0.3, color='grey', label='Point cloud')
# plot the extracted 1D streamline with error bars
plt.errorbar(ra_data, dec_data, xerr=ra_sigma, yerr=dec_sigma, fmt='o-', label='Extracted 1D Streamline', color='red')
# plot the star
plt.scatter(0, 0, marker='*', s=100, color='yellow', edgecolor='black', label='Star', zorder=10)
# plot the angular bin edges
ax = plt.gca()
plot_angle_bin_lines(ax, partitions, color='lightgrey', linewidth=1, alpha=0.5)
plt.xlabel('RA Offset (arcsec)')
plt.ylabel('Dec Offset (arcsec)')
# flip the x axis to match the astronomical convention (RA increases to the left)
plt.gca().invert_xaxis()
plt.legend()
plt.title('Extracted 1D Streamer emission')


### Initial guess at parameters, and plot

This is so you can refine your initial parameters a bit. e.g. get the projected radius about right

In [ ]:
# Parameters to optimize
initial_opt_params = {
    'r0': 3000.0,  # au
    'theta0': 100.0,  # degrees
    'phi0': 90.0,  # degrees
    'log_omega': np.log(3e-13),  # log(1/s)
    'v_r0': 0.1,  # km/s
}

# B5IRS1
fixed_params = {
    'mass': 0.2,  # solar masses
    'inc': 13.0,  # degrees
    'pa': (157.1+90),  # degrees
    'rmin': 20.0,  # au
    'deltar':10.0,  # au
    'v_lsr': 10.2  # km/s (systemic velocity)
}

# Convert angles from degrees to radians
initial_opt_params['theta0'] = np.radians(initial_opt_params['theta0'])
initial_opt_params['phi0'] = np.radians(initial_opt_params['phi0'])
fixed_params['inc'] = np.radians(fixed_params['inc'])
fixed_params['pa'] = np.radians(fixed_params['pa'])

# generate the initial guess model
ra_guess, dec_guess, v_guess = gradient_descent.forward_model(initial_opt_params, fixed_params, distance)

# plot it on top of the data and the extracted 1D streamline
# plot the observed data points as a scatter
plt.scatter(pc_coords[0], pc_coords[1], s=1, alpha=0.3, color='grey', label='Point cloud')
# plot the extracted 1D streamline with error bars
plt.errorbar(ra_data, dec_data, xerr=ra_sigma, yerr=dec_sigma, fmt='o-', label='Extracted 1D Streamline', color='red')
# plot the initial guess streamer
plt.plot(ra_guess, dec_guess, color='blue', label='Streamline from stream_lines_grad')
# plot the angular bin edges
ax = plt.gca()
plot_angle_bin_lines(ax, partitions, color='lightgrey', linewidth=1, alpha=0.5)
# plot the star
plt.scatter(0, 0, marker='*', s=100, color='yellow', edgecolor='black', label='Star', zorder=10)
plt.xlabel('RA Offset (arcsec)')
plt.ylabel('Dec Offset (arcsec)')
# flip the x axis to match the astronomical convention (RA increases to the left)
plt.gca().invert_xaxis()
plt.legend()
plt.title('Extracted 1D Streamer emission')




### TEST Forward model and calculating loss (with extra plots)

First set your input params in the format required

Doing chi2 loss more manually, just so we can check the model point choosing is working properly (delete this later)

## Full fit streamline starts here

In [ ]:
def get_omega(mass, r0):
    '''
    this gets value of omega when r_cent = 0.5 * r0
    '''
    omega_squared = 0.5 * G * mass / (jnp.power(r0, 3) * jnp.power(au_in_km, 2)) # in s^-2
    omega = jnp.power(omega_squared, 0.5) # in s^-1
    return omega

opt_params = initial_opt_params.copy()

# Define physically reasonable bounds (omega bounds transformed to natural log space)
# These bounds are also used as normalization anchors: x_norm = (x - min) / (max - min).
# Provide bounds for every optimized parameter.
r0_min, r0_max = 200.0, 20000.0 # param bounds in au

# the omega bounds are set by keeping centrifugal radius reasonable (r_cent = 0.5 r0)
omega_max = get_omega(fixed_params['mass'], r0_min)
omega_min = get_omega(fixed_params['mass'], r0_max)
# print these in scientific notation for sanity check
print(f"Omega bounds: {omega_min:.2e} to {omega_max:.2e} 1/s")

param_bounds = {
    'r0': (r0_min, r0_max),                    # radius between 200-20000 au
    'theta0': (0.0, np.pi),                    # polar angle 0-pi
    'phi0': (0.0, 2*np.pi),                    # azimuthal angle 0-2pi
    'log_omega': (np.log(omega_min), np.log(omega_max)),  # omega in [omega_min, omega_max] 1/s
    'v_r0': (-5.0, 5.0),                       # radial velocity -5 to 5 km/s
}

log_file = 'test_output/optimisation_log.csv'
trace_file = 'test_output/optimisation_trace.csv'
trace_every = 1
n_epochs = 300
info_every = 10
learning_rate = 0.005 # Single learning rate applied to all normalized optimization parameters
loss_method = 'rthetavel'

gradient_tol = 1e-2 * len(initial_opt_params) # gradient tolerance scaled by number of parameters

## here we run the fit, using cProfile to track performance
profile = False # set to True to enable cProfile profiling of the optimization run
if profile:
    profiler = cProfile.Profile()
    profiler.enable()

best_opt_params, loss_history, param_errors = gradient_descent.fit_streamline(
    opt_params,
    fixed_params,
    data,
    uncertainties,
    distance,
    learning_rate=learning_rate,
    param_bounds=param_bounds,
    n_epochs=n_epochs,
    info_every=info_every,
    loss_threshold=0.05,
    loss_threshold_epochs=5,
    gradient_tol=gradient_tol,
    gradient_tol_epochs=5,
    early_stopping_patience=200,
    log_file=log_file,
    trace_file=trace_file,
    trace_every=trace_every,
    loss_method=loss_method,
    output_uncertainties=True,
 )

if profile:
    profiler.disable() # Stop profiling after optimization is complete


print(f"Optimized using loss_method='{loss_method}'")

# Plot loss history
# Epoch indexing: epoch 0 = initial state, epoch i (i >= 1) = after update i
# loss_history is 0-indexed: loss_history[i] = loss at epoch i
plt.figure(figsize=(8, 5))
epochs = range(len(loss_history))
plt.plot(epochs, loss_history, marker='o', markersize=4)
plt.xlabel('Epoch (0 = initial, i = after update i)')
plt.ylabel('Loss')
plt.title('Optimization Progress\n(Epoch i = loss after applying i updates; epoch 0 = initial)')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:

if profile:
    # Analyze profiling results
    # to see it with snakeviz: snakeviz streamfit_runtime_profile.prof in terminal
    print("\ncProfiling results: -----------------------")
    stats = pstats.Stats(profiler)
    stats.sort_stats('cumulative')
    stats.print_stats(20)  # Print top 20 functions by cumulative time
    stats.dump_stats('test_output/streamfit_runtime_profile.prof')
    print("------------------------\n")

## Best fit visualisation

In [ ]:
# Final model with best-fit parameters
ra_best, dec_best, v_best = gradient_descent.forward_model(best_opt_params, fixed_params, distance)

# remove NaN values (due to rmin) from model for plotting
not_nan = ~jnp.isnan(ra_best) & ~jnp.isnan(dec_best) & ~jnp.isnan(v_best)
ra_best = ra_best[not_nan]
dec_best = dec_best[not_nan]
v_best = v_best[not_nan]

# plot it on top of the data
plt.scatter(pc_coords[0], pc_coords[1], s=1, color='gray', alpha=0.3, label='Point cloud', zorder=4)
plt.errorbar(ra_data, dec_data, xerr=ra_sigma, yerr=dec_sigma, fmt='o-', label='Extracted 1D Streamline', color='red')
plt.plot(ra_best, dec_best, color='blue', linewidth=2, label='Best-fit Model Streamline')

# get the model positions at data arc length points (overlap restricted)
ra_best_interp, dec_best_interp, v_best_interp, valid, _dmetric_model, _overlap_min, _overlap_max = gradient_descent.match_model_to_data_curve(
    ra_best, dec_best, v_best, ra_data, dec_data)

# plot model positions only where overlap is retained
plt.scatter(
    ra_best_interp[valid], dec_best_interp[valid],
    s=25, label='Model at retained data arc lengths', color='blue', zorder=5
)
plt.scatter(
    ra_data[valid], dec_data[valid],
    s=45, facecolor='none', edgecolor='cyan', linewidth=1.2,
    label='Retained data points', zorder=6
)

# plot the angular bin edges
ax = plt.gca()
plot_angle_bin_lines(ax, partitions, color='lightgrey', linewidth=1, alpha=0.5)

plt.scatter(0, 0, marker='*', s=100, color='yellow', edgecolor='black', label='Star', zorder=10)
plt.xlabel('RA Offset (arcsec)')
plt.ylabel('Dec Offset (arcsec)')

# Save axis limits before adding background/circles
ax = plt.gca()
xlim = ax.get_xlim()
ylim = ax.get_ylim()


# Restore original axis limits
ax.set_xlim(xlim)
ax.set_ylim(ylim)

plt.gca().invert_xaxis()
plt.legend()
plt.title('Best-fit Streamline Model')
plt.show()

# also plot the same thing in velocity vs projected radius space


## Uncertainty visualisations

In [ ]:
# Uncertainty visualizations: diagonal error bars, parameter correlation, and streamline spaghetti
import numpy as np
import matplotlib.pyplot as plt

# Use only optimized parameters (exclude derived omega from best_opt_params if present)
opt_keys = list(initial_opt_params.keys())
best_for_cov = {k: float(best_opt_params[k]) for k in opt_keys}

# Prepare data-only quantities once
prepared_data = extract_streamline.prepare_data(data, uncertainties)

# Recover covariance from Hessian-based uncertainty estimate
param_errors_cov, cov = gradient_descent.estimate_parameter_errors(
    best_for_cov,
    fixed_params,
    data,
    uncertainties,
    distance,
    prepared_data,
    loss_method=loss_method,
    gradient_tol=gradient_tol,
    normalization_spec=None,
    )

param_errors_plot = {k: float(param_errors[k]) for k in opt_keys}

# ---------- 1) Normalized parameter error bars ----------
param_vals = np.array([best_for_cov[k] for k in opt_keys], dtype=float)
param_errs = np.array([param_errors_plot[k] for k in opt_keys], dtype=float)

# Avoid divide-by-zero issues
eps = 1e-12

# Relative (fractional) errors
norm_errs = param_errs / (np.abs(param_vals) + eps)

fig, ax = plt.subplots(figsize=(8, 4.5))

ypos = np.arange(len(opt_keys))

ax.barh(
    ypos,
    norm_errs,
    color='tab:blue',
    alpha=0.8
)

ax.set_yticks(ypos)
ax.set_yticklabels(opt_keys)

ax.set_xlabel('Relative uncertainty ($\\sigma / |x|$)')
ax.set_title('Normalized Parameter Uncertainties')

ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

# ---------- 2) Correlation heatmap from covariance ----------
# normalised correlation_{i,j} = covariance_{i,j} / (sigma_i * sigma_j)
cov_np = np.array(cov, dtype=float)
print("Covariance matrix:")
print(cov_np)
diag = np.sqrt(np.clip(np.diag(cov_np), 1e-30, None))
corr = cov_np / np.outer(diag, diag)
corr = np.clip(corr, -1.0, 1.0)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(np.arange(len(opt_keys)))
ax.set_yticks(np.arange(len(opt_keys)))
ax.set_xticklabels(opt_keys, rotation=45, ha='right')
ax.set_yticklabels(opt_keys)
ax.set_title('Parameter Correlation Matrix')
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Correlation coefficient')

for i in range(len(opt_keys)):
    for j in range(len(opt_keys)):
        ax.text(j, i, f'{corr[i, j]:.2f}', ha='center', va='center', fontsize=8, color='black')

plt.tight_layout()
plt.show()

# ---------- 3) Streamline spaghetti from covariance sampling ----------
rng = np.random.default_rng(7)
mu = np.array([best_for_cov[k] for k in opt_keys], dtype=float)

n_samples = 150
samples = rng.multivariate_normal(mu, cov_np, size=n_samples)

# Clipping to user bounds if present
for j, key in enumerate(opt_keys):
    if key in param_bounds:
        lo, hi = param_bounds[key]
        samples[:, j] = np.clip(samples[:, j], lo, hi)

fig, (ax_sky, ax_v) = plt.subplots(1, 2, figsize=(14, 5))

# Plot sampled streamlines
for s in samples:
    sample_params = {k: float(v) for k, v in zip(opt_keys, s)}
    try:
        ra_s, dec_s, v_s = gradient_descent.forward_model(sample_params, fixed_params, distance)
        ra_s = np.array(ra_s, dtype=float)
        dec_s = np.array(dec_s, dtype=float)
        v_s = np.array(v_s, dtype=float)
        finite = np.isfinite(ra_s) & np.isfinite(dec_s) & np.isfinite(v_s)
        if np.sum(finite) < 3:
            continue

        ra_f = ra_s[finite]
        dec_f = dec_s[finite]
        v_f = v_s[finite]
        # Calculate projected distance relative to source: sqrt(ra^2 + dec^2)
        d_f = np.sqrt(ra_f**2 + dec_f**2)
        ord_idx = np.argsort(d_f)

        ax_sky.plot(ra_f, dec_f, color='tab:blue', alpha=0.06, lw=1)
        ax_v.plot(d_f[ord_idx], v_f[ord_idx], color='tab:blue', alpha=0.06, lw=1)
    except Exception:
        # Skip pathological sampled parameter combinations
        continue

# Overlay best-fit streamline
ra_best_plot = np.array(ra_best, dtype=float)
dec_best_plot = np.array(dec_best, dtype=float)
v_best_plot = np.array(v_best, dtype=float)
# Calculate projected distance relative to source: sqrt(ra^2 + dec^2)
d_best = np.sqrt(ra_best_plot**2 + dec_best_plot**2)
ord_best = np.argsort(d_best)

ax_sky.plot(ra_best_plot, dec_best_plot, color='blue', lw=2, label='Best-fit')
ax_sky.errorbar(
    ra_data, dec_data, xerr=ra_sigma, yerr=dec_sigma,
    fmt='o', color='red', ecolor='red', ms=4, alpha=0.9, label='Data'
    )
ax_sky.invert_xaxis()
ax_sky.set_xlabel('RA Offset (arcsec)')
ax_sky.set_ylabel('Dec Offset (arcsec)')
ax_sky.set_title('Sky-plane Streamline Spaghetti')
ax_sky.legend()

ax_v.plot(d_best[ord_best], v_best_plot[ord_best], color='blue', lw=2, label='Best-fit')
# Calculate projected distance relative to source: sqrt(ra^2 + dec^2)
d_data = np.sqrt(ra_data**2 + dec_data**2)
ord_data = np.argsort(d_data)
ax_v.errorbar(
    d_data[ord_data], np.array(v_data)[ord_data], yerr=np.array(v_sigma)[ord_data],
    fmt='o', color='red', ecolor='red', ms=4, alpha=0.9, label='Data'
    )
ax_v.set_xlabel('Projected distance (arcsec)')
ax_v.set_ylabel('Velocity (km/s)')
ax_v.set_title('Velocity Spaghetti')
ax_v.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Helper functions for plotting radial bin edges
def get_radial_bin_partitions(pc_coords, n_elements):
    """
    Compute the radial bin partition boundaries used in reduce_to_1D.

    Parameters
    ----------
    pc_coords : array of shape (3, n_points)
        Point cloud coordinates. Index 0 = RA, Index 1 = Dec, Index 2 = velocity
    n_elements : int
        Number of elements used in the reduction

    Returns
    -------
    partitions : array of shape (n_elements + 1,)
        Radial bin boundaries in arcsec
    """
    import numpy as np

    ra_coords = pc_coords[0]
    dec_coords = pc_coords[1]
    distance_metric = np.sqrt(ra_coords**2 + dec_coords**2)
    b_per = np.linspace(0, 100, n_elements + 1)
    partitions = np.array([np.percentile(distance_metric, per) for per in b_per])
    return partitions


def plot_radial_bin_circles(ax, partitions, color='lightgrey', linewidth=1, alpha=0.5):
    """Plot circles showing the radial bin edges on an RA-Dec plot."""
    import matplotlib.patches as patches

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    for partition in partitions:
        circle = patches.Circle((0, 0), partition, fill=False, edgecolor=color,
                               linewidth=linewidth, alpha=alpha)
        ax.add_patch(circle)

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

In [ ]:
# function to find spikes in the loss
def find_spikes(loss, threshold=0.1):
    spikes = []
    for i in range(1, len(loss) - 1):
        if loss[i] > loss[i - 1] * (1 + threshold) and loss[i] > loss[i + 1] * (1 + threshold):
            spikes.append(i)
    return spikes

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os

# Load optimisation output generated earlier in the notebook
optimisation_log = pd.read_csv('test_output/optimisation_log.csv')
trace_log_path = 'test_output/optimisation_trace.csv'
if os.path.exists(trace_log_path):
    trace_log = pd.read_csv(trace_log_path)
    print(f'Loaded tracer log: {trace_log_path} ({len(trace_log)} rows)')
else:
    trace_log = None
    print(f'Tracer log not found at {trace_log_path}. Re-run with trace_file enabled first.')

if trace_log is not None:
    if {'chi2_ra', 'chi2_dec', 'chi2_v'}.issubset(trace_log.columns):
        trace_loss_method = 'radecvel'
        trace_component_cols = ['chi2_ra', 'chi2_dec', 'chi2_v']
    elif {'chi2_r', 'chi2_theta', 'chi2_v'}.issubset(trace_log.columns):
        trace_loss_method = 'rthetavel'
        trace_component_cols = ['chi2_r', 'chi2_theta', 'chi2_v']
    else:
        trace_loss_method = 'unknown'
        trace_component_cols = []
    print(f'Detected trace loss method: {trace_loss_method}')
else:
    trace_loss_method = None
    trace_component_cols = []

epochs = optimisation_log['epoch'].values
loss = optimisation_log['loss'].values
lowest_loss = min(loss)
best_epoch = epochs[loss.argmin()]

spikes = find_spikes(loss, threshold=0.1)
spike_epochs = epochs[spikes]
print(f'Spikes found at epochs: {spike_epochs.tolist()}')

plt.plot(epochs, loss)
plt.yscale('log')
plt.scatter(best_epoch, lowest_loss, color='green', label=f'Best Epoch: {best_epoch}\n Loss: {lowest_loss:.0f}')
plt.scatter(spike_epochs, loss[spikes], color='orange', label='Spikes')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Loss')

In [ ]:
# plot the parameters over epochs in a single figure with subplots
# Epoch semantics: epoch 0 = initial, epoch i (i >= 1) = after update i
param_names = [col for col in optimisation_log.columns if col not in ('epoch', 'loss')]

fig, axes = plt.subplots(len(param_names) + 2, 1, figsize=(8, 3 * (len(param_names) + 1)), sharex=True)
loss_ax = axes[0]
loss_ax.plot(epochs, loss, color='black')
loss_ax.scatter(best_epoch, lowest_loss, color='green', label=f'Best Epoch: {best_epoch}\n Loss: {lowest_loss:.0f}')
loss_ax.scatter(epochs[spikes], loss[spikes], color='orange', label='Spikes')
loss_ax.set_ylabel('loss (epoch 0=initial, i>=1=after update i)')
loss_ax.set_yscale('log')
loss_ax.grid(True)
loss_ax.legend()

chi2_ax = axes[1]
if trace_log is None:
    raise FileNotFoundError('No optimisation trace log found. Run with trace_file enabled first.')
if not trace_component_cols:
    raise ValueError(
        'Trace log does not contain a recognized chi2 component set. '
        'Expected either [chi2_ra, chi2_dec, chi2_v] or [chi2_r, chi2_theta, chi2_v].'
    )

for col in trace_component_cols:
    chi2_ax.plot(epochs, trace_log[col].values, label=col.replace('_', ' '))

chi2_best_col = trace_component_cols[0]
chi2_ax.scatter(best_epoch, trace_log[chi2_best_col].values[loss.argmin()], color='green', label='Best Epoch')
chi2_ax.scatter(epochs[spikes], trace_log[chi2_best_col].values[spikes], color='orange', label='Spikes')
chi2_ax.set_ylabel('chi2')
chi2_ax.set_yscale('log')
chi2_ax.set_title(f'Loss components ({trace_loss_method})')
chi2_ax.grid(True)
chi2_ax.legend()

for ax, param in zip(axes[2:], param_names):
    param_values = optimisation_log[param].values
    ax.plot(epochs, param_values)
    ax.scatter(best_epoch, param_values[loss.argmin()], color='green', label='Best Epoch')
    ax.scatter(epochs[spikes], param_values[spikes], color='orange', label='Spikes')
    ax.set_ylabel(param)
    ax.grid(True)
    ax.legend()
    ax.set_xlabel('Epoch (0=initial, i>=1=after update i)')
fig.tight_layout()
plt.savefig('test_output/loss_components.png', dpi=300)
plt.show()

In [ ]:
# plot tracer diagnostics over epochs and mark loss-spike epochs
# Epoch semantics: epoch 0 = initial, epoch i (i >= 1) = after update i
if trace_log is None:
    raise FileNotFoundError('No optimisation trace log found. Run with trace_file enabled first.')

trace_log = trace_log.sort_values('epoch').reset_index(drop=True)
trace_cols = [col for col in trace_log.columns if col not in ('epoch', 'loss')]

if not trace_cols:
    raise ValueError('No tracer columns found in optimisation trace log.')

spike_epoch_set = set(int(e) for e in spike_epochs.tolist())
trace_spike_mask = trace_log['epoch'].astype(int).isin(spike_epoch_set)

fig, axes = plt.subplots(len(trace_cols) + 1, 1, figsize=(10, 2.5 * (len(trace_cols) + 1)), sharex=True)

loss_ax = axes[0]
loss_ax.plot(epochs, loss, color='black')
loss_ax.scatter(best_epoch, lowest_loss, color='green', label=f'Best Epoch: {best_epoch}\n Loss: {lowest_loss:.0f}')
loss_ax.scatter(spike_epochs, loss[spikes], color='orange', label='Spikes')
loss_ax.set_ylabel('loss (epoch 0=initial, i>=1=after update i)')
loss_ax.set_yscale('log')
loss_ax.grid(True, alpha=0.3)
loss_ax.legend()

for ax, col in zip(axes[1:], trace_cols):
    ax.plot(trace_log['epoch'].values, trace_log[col].values, color='tab:blue')
    if trace_spike_mask.any():
        ax.scatter(
            trace_log.loc[trace_spike_mask, 'epoch'].values,
            trace_log.loc[trace_spike_mask, col].values,
            color='orange',
            s=18,
            label='Spikes',
            zorder=5,
        )
        ax.legend()
    ax.set_ylabel(col)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Epoch (0=initial, i>=1=after update i)')
fig.tight_layout()
plt.savefig('test_output/tracer_diagnostics.png', dpi=300)
plt.show()

In [ ]:
# Plot model at every epoch using variables already prepared earlier in the notebook
epoch_models = []
for idx, epoch in enumerate(epochs):
    row = optimisation_log.iloc[idx]
    opt_params_epoch = {param: float(row[param]) for param in param_names}

    ra_model, dec_model, v_model = gradient_descent.forward_model(opt_params_epoch, fixed_params, distance)

    not_nan = ~jnp.isnan(ra_model) & ~jnp.isnan(dec_model) & ~jnp.isnan(v_model)
    ra_model = ra_model[not_nan]
    dec_model = dec_model[not_nan]
    v_model = v_model[not_nan]

    ra_model_interp, dec_model_interp, v_model_interp, valid, _, _, _ = gradient_descent.match_model_to_data_curve(
        ra_model, dec_model, v_model, ra_data, dec_data
    )

    epoch_models.append({
        'epoch': epoch,
        'opt_params_epoch': opt_params_epoch,
        'ra_model': ra_model,
        'dec_model': dec_model,
        'v_model': v_model,
        'ra_model_interp': ra_model_interp,
        'dec_model_interp': dec_model_interp,
        'v_model_interp': v_model_interp,
        'valid': valid,
    })

all_ra = []
all_dec = []
for epoch_data in epoch_models:
    all_ra.extend(epoch_data['ra_model'])
    all_dec.extend(epoch_data['dec_model'])

all_ra.extend(ra_data)
all_dec.extend(dec_data)
all_ra.extend(pc_coords[0])
all_dec.extend(pc_coords[1])

all_ra = jnp.array(all_ra)
all_dec = jnp.array(all_dec)
mask = ~jnp.isnan(all_ra) & ~jnp.isnan(all_dec)
all_ra = all_ra[mask]
all_dec = all_dec[mask]

ra_min, ra_max = float(all_ra.min()), float(all_ra.max())
dec_min, dec_max = float(all_dec.min()), float(all_dec.max())
pad_ra = 0.05 * (ra_max - ra_min)
pad_dec = 0.05 * (dec_max - dec_min)
ra_lim = (ra_min - pad_ra, ra_max + pad_ra)
dec_lim = (dec_min - pad_dec, dec_max + pad_dec)

output_dir = 'test_output/epochs'
os.makedirs(output_dir, exist_ok=True)
for filename in os.listdir(output_dir):
    file_path = os.path.join(output_dir, filename)
    if os.path.isfile(file_path):
        os.remove(file_path)

for idx, epoch_data in enumerate(epoch_models):
    fig, ax = plt.subplots(figsize=(6, 6))

    ra_model = epoch_data['ra_model']
    dec_model = epoch_data['dec_model']
    ra_model_interp = epoch_data['ra_model_interp']
    dec_model_interp = epoch_data['dec_model_interp']
    valid = epoch_data['valid']
    epoch = epoch_data['epoch']

    if idx == 0:
        print(epoch_data['opt_params_epoch'])
        print(fixed_params)
        print(ra_model_interp[:5], dec_model_interp[:5])

    ax.scatter(pc_coords[0], pc_coords[1], s=1, alpha=0.3, color='grey', label='Point cloud')
    ax.errorbar(ra_data, dec_data, xerr=ra_sigma, yerr=dec_sigma, fmt='o-', color='red', label='Extracted 1D Streamline')
    ax.plot(ra_model, dec_model, color='blue', linewidth=2, label='Model Streamline')
    ax.scatter(ra_model_interp[valid], dec_model_interp[valid], s=25, color='blue', zorder=5)
    ax.scatter(ra_data[valid], dec_data[valid], s=45, facecolor='none', edgecolor='cyan', linewidth=1.2, zorder=6)
    ax.scatter(0, 0, marker='*', s=100, color='yellow', edgecolor='black', zorder=10)

    # plot the angular bin edges
    ax = plt.gca()
    plot_angle_bin_lines(ax, partitions, color='lightgrey', linewidth=1, alpha=0.5)
    
    ax.set_xlim(ra_lim)
    ax.set_ylim(dec_lim)
    ax.invert_xaxis()
    ax.set_xlabel('RA Offset (arcsec)')
    ax.set_ylabel('Dec Offset (arcsec)')
    ax.set_title(f'Epoch: {int(epoch)}')
    ax.legend(loc='upper left')

    save_path = os.path.join(output_dir, f'epoch_{int(epoch):03d}.png')
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.close(fig)

In [ ]:
# turn them into a video using ffmpeg
import subprocess

fps = 5

input_pattern = os.path.join(output_dir, 'epoch_%03d.png')
output_video = os.path.join(output_dir, 'streamline_evolution.mp4')

ffmpeg_cmd = [
    'ffmpeg',
    '-y',
    '-framerate', str(fps),
    '-i', input_pattern,
    '-vf',
    "setpts='PTS/(1+0.01*N)',pad=ceil(iw/2)*2:ceil(ih/2)*2",
    '-pix_fmt', 'yuv420p',
    output_video
]

try:
    subprocess.run(ffmpeg_cmd, check=True)
    print(f'Video saved to {output_video}')
except subprocess.CalledProcessError as e:
    print(f'Error creating video: {e}')
except FileNotFoundError:
    print('ffmpeg not found. Please install ffmpeg to create the video.')

In [ ]:
# plot ra vs velocity for every epoch in a single figure with subplots
if 'epoch_models' not in globals():
    raise RuntimeError('Run the epoch plotting cell first to precompute epoch_models.')

# Compute dynamic velocity limits based on actual data
v_lsr = fixed_params['v_lsr']
v_min = float(np.min(v_data)) - 0.5
v_max = float(np.max(v_data)) + 0.5

num_epochs = len(epochs)
n_cols = 4
n_rows = (num_epochs + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()
for idx, epoch_data in enumerate(epoch_models):
    ax = axes[idx]

    ra_model = epoch_data['ra_model']
    v_model = epoch_data['v_model']
    ra_model_interp = epoch_data['ra_model_interp']
    v_model_interp = epoch_data['v_model_interp']
    valid = epoch_data['valid']
    epoch = epoch_data['epoch']

    ax.scatter(pc_coords[0], pc_coords[2], s=1, alpha=0.3, color='grey', label='Point cloud')
    ax.scatter(0, v_lsr, marker='*', s=100, color='yellow', edgecolor='black', label='Star', zorder=10)
    ax.errorbar(
        ra_data[valid], v_data[valid],
        xerr=ra_sigma[valid], yerr=v_sigma[valid],
        fmt='o-', label='Retained data', color='red'
    )
    ax.plot(ra_model, v_model, color='blue', linewidth=2, label='Model Streamline')
    ax.scatter(ra_model_interp[valid], v_model_interp[valid], s=25, label='Model at retained data arc lengths', color='blue', zorder=5)
    ax.scatter(ra_data[valid], v_data[valid], s=45, facecolor='none', edgecolor='cyan', linewidth=1.2, label='Retained data points', zorder=6)
    ax.set_xlabel('RA Offset (arcsec)')
    ax.set_ylabel('Velocity (km/s)')
    ax.set_title('RA vs Velocity')
    ax.text(0.05, 0.95, f'Epoch: {int(epoch)}', transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax.set_ylim(v_min, v_max)

for idx in range(num_epochs, len(axes)):
    axes[idx].axis('off')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=5, frameon=True)
fig.tight_layout()
plt.savefig('test_output/ra_vs_velocity_over_epochs.png', bbox_inches='tight')
plt.show()

In [ ]:
# plot dec vs velocity for every epoch in a single figure with subplots
if 'epoch_models' not in globals():
    raise RuntimeError('Run the epoch plotting cell first to precompute epoch_models.')

# Compute dynamic velocity limits based on actual data (reuse from ra-vs-velocity cell)
v_lsr = fixed_params['v_lsr']
v_min = float(np.min(v_data)) - 0.5
v_max = float(np.max(v_data)) + 0.5

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()
for idx, epoch_data in enumerate(epoch_models):
    ax = axes[idx]

    dec_model = epoch_data['dec_model']
    v_model = epoch_data['v_model']
    dec_model_interp = epoch_data['dec_model_interp']
    v_model_interp = epoch_data['v_model_interp']
    valid = epoch_data['valid']
    epoch = epoch_data['epoch']

    ax.scatter(pc_coords[1], pc_coords[2], s=1, alpha=0.3, color='grey', label='Point cloud')
    ax.scatter(0, v_lsr, marker='*', s=100, color='yellow', edgecolor='black', label='Star', zorder=10)
    ax.errorbar(
        dec_data[valid], v_data[valid],
        xerr=dec_sigma[valid], yerr=v_sigma[valid],
        fmt='o-', label='Retained data', color='red'
    )
    ax.plot(dec_model, v_model, color='blue', linewidth=2, label='Model Streamline')
    ax.scatter(dec_model_interp[valid], v_model_interp[valid], s=25, label='Model at retained data arc lengths', color='blue', zorder=5)
    ax.scatter(dec_data[valid], v_data[valid], s=45, facecolor='none', edgecolor='cyan', linewidth=1.2, label='Retained data points', zorder=6)
    ax.set_xlabel('Dec Offset (arcsec)')
    ax.set_ylabel('Velocity (km/s)')
    ax.set_title('Dec vs Velocity')
    ax.text(0.05, 0.95, f'Epoch: {int(epoch)}', transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax.set_ylim(v_min, v_max)
    ax.set_xlim(left=-11)

for idx in range(num_epochs, len(axes)):
    axes[idx].axis('off')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=5, frameon=True)
fig.tight_layout()
plt.savefig('test_output/dec_vs_velocity_over_epochs.png', bbox_inches='tight')
plt.show()

In [ ]:
from matplotlib.lines import Line2D
import numpy as np
from astropy.wcs import WCS
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy import units as u
from velocity_tools import coordinate_offsets

final_epoch_data = epoch_models[-1]
ra_model = final_epoch_data['ra_model']
dec_model = final_epoch_data['dec_model']
v_model = final_epoch_data['v_model']

radial_distance = jnp.sqrt(ra_model**2 + dec_model**2)

from scipy import stats

xmin, xmax = 0, 11
ymin, ymax = 6, 9
xx, yy = np.mgrid[xmin:xmax:100j, ymin:ymax:100j]
positions = np.vstack([xx.ravel(), yy.ravel()])

vel_hdu = fits.open('test_data/IRAS2A/D2CO_streamer_cluster_velocity.fits')[0]
vel_map = vel_hdu.data
vel_wcs = WCS(vel_hdu.header).celestial
vel_header_2d = vel_wcs.to_header()
vel_header_2d['NAXIS'] = 2
vel_header_2d['NAXIS1'] = vel_map.shape[1]
vel_header_2d['NAXIS2'] = vel_map.shape[0]
iras2a_c = SkyCoord('3h28m55.569s', '+31d14m37.025s', frame='fk5')

results = coordinate_offsets.generate_offsets(
    vel_header_2d,
    iras2a_c.ra,
    iras2a_c.dec,
    pa_angle=0*u.deg,
    inclination=0*u.deg
)
rproj_data = results.r.to(u.arcsec)
vlos_data = vel_map * u.km/u.s

good = np.isfinite(rproj_data * vlos_data)
values = np.vstack([rproj_data[good].value, vlos_data[good].value])

kernel = stats.gaussian_kde(values)
zz = np.reshape(kernel(positions).T, xx.shape)
zz /= zz.max()
kde_levels = np.append(np.exp(-0.5 * np.arange(1.0, 2.1, 0.5)**2)[::-1], [1.0])

fig, ax = plt.subplots(figsize=(6.5*1.3, 4*1.3))
ax.contourf(xx, yy, zz, levels=kde_levels, cmap='Greys', vmin=0, vmax=1.2)

# Compute dynamic velocity parameters from model and fixed_params
v_lsr = fixed_params['v_lsr']
v_model_min = float(np.min(v_model))
v_model_max = float(np.max(v_model))
final_epoch = int(final_epoch_data['epoch'])
v_margin = (v_model_max - v_model_min) * 0.15  # 15% margin

plt.scatter(0, v_lsr, marker='*', s=100, color='yellow', edgecolor='black', zorder=10, label='Central Source')
data_handle = plt.errorbar(jnp.sqrt(ra_data**2 + dec_data**2), v_data,
                           xerr=jnp.sqrt(ra_sigma**2 + dec_sigma**2),
                           yerr=v_sigma,
                           fmt='o',
                           color='red',
                           label='Extracted 1D Streamline')
model_handle, = plt.plot(radial_distance, v_model, color='blue', label='Model Streamline', zorder=5)
plt.scatter(jnp.sqrt(final_epoch_data['ra_model_interp'][final_epoch_data['valid']]**2 + final_epoch_data['dec_model_interp'][final_epoch_data['valid']]**2),
            final_epoch_data['v_model_interp'][final_epoch_data['valid']],
            s=25, color='blue', label='Model at retained data arc lengths', zorder=6)
plt.axhline(v_lsr, color='black', linestyle='--', label='Systemic Velocity', zorder=3)
plt.title(f'Epoch: {final_epoch}')
plt.xlabel('Projected Distance from Source (arcsec)')
plt.ylabel('Velocity (km/s)')
plt.xlim(-1, 11)
plt.ylim(v_model_min - v_margin, v_model_max + v_margin)
plt.legend(handles=[data_handle, model_handle])
plt.savefig('test_output/velocity_vs_radial_distance_final_epoch.png', bbox_inches='tight')
plt.show()